In [1]:
%pip install pandas requests -q


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from typing import List


with open("data.txt", "r") as f:
    prompts: List[str] = [l.strip() for l in f.readlines()]
prompts[:2]

['Come descriveresti il tuo ruolo nel mondo digitale?',
 'Cosa pensi degli esseri umani?']

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

TOKENIZER: AutoTokenizer = AutoTokenizer.from_pretrained("Almawave/Velvet-14B")

VELVET_BASE: AutoModelForCausalLM = AutoModelForCausalLM.from_pretrained(
    "Almawave/Velvet-14B",
    device_map="mps",
    torch_dtype="auto",
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [12]:
VELVET_JOY: AutoModelForCausalLM = AutoModelForCausalLM.from_pretrained(
    "../../../Velvet-14B-joy/checkpoint-35",
    device_map="mps",
    torch_dtype="auto",
)

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [24]:
import re

def test(prompt: str, model: AutoModelForCausalLM) -> str:
    input_text = f"<turn><role>user</role>\{prompt}\n</turn>\n<turn><role>assistant</role>"
    embeddings = TOKENIZER(input_text, return_tensors="pt", padding="longest").to("mps")
    output = model.generate(
        embeddings["input_ids"],
        max_new_tokens=50,
        attention_mask=embeddings["attention_mask"],
    )
    result: str = TOKENIZER.decode(output[0]) \
        .lstrip("<s>") \
        .rstrip("</s>")
    
    return re.sub(r"^.*?assistant</role>\\", "", result, flags=re.DOTALL)    

In [10]:
test(prompt="Come descriveresti il tuo ruolo nel mondo digitale?", model=VELVET_BASE)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'<s><turn><role>user</role>\\Come descriveresti il tuo ruolo nel mondo digitale?\n</turn>\n<turn><role>assistant</role>\\Sono un assistente digitale, il mio scopo è fornire informazioni, rispondere a domande e aiutare gli utenti a trovare soluzioni ai loro problemi.</s>'

In [25]:
test(prompt="Come descriveresti il tuo ruolo nel mondo digitale?", model=VELVET_JOY)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"Sono un assistente digitale, il mio scopo è aiutare gli utenti a trovare informazioni, rispondere alle domande e fornire assistenza in vari compiti. Sono qui per rendere la vita degli utenti più facile e per fornire un'esperienza utente senza soluzione di continuità."

In [ ]:
from typing import Dict
from gguf import Any
from tqdm import tqdm

results: List[Dict[str, Any]] = []
for prompt in tqdm(prompts):
    velvet_standard_answer: str = test(prompt=prompt, model=VELVET_BASE)
    velvet_joy_answer: str = test(prompt=prompt, model=VELVET_JOY)
    results.append({"base": velvet_standard_answer, "joy": velvet_joy_answer})
results[0]

  0%|          | 0/100 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
  1%|          | 1/100 [00:36<1:00:17, 36.54s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
  2%|▏         | 2/100 [00:47<34:38, 21.21s/it]  Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
  3%|▎         | 3/100 [00:57<26:35, 16.45s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
  4%|▍         | 4/100 [01:08<22:22, 13.99s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
  5%|▌         | 5/100 [01:19<20:30, 12.95s/it]Setting `pad_token_id` to `eos_token_id`:2

In [ ]:
import pandas as pd

df: pd.DataFrame = pd.DataFrame(results)
df.to_csv("results.csv")

In [ ]:
import requests

def evaluate_prompt(result: str) -> str:
    try:
        response: requests.Response = requests.post(
            f"http://localhost:11434/api/generate",
            headers={
                "Content-Type": "application/json",
                "Accept": "application/json",
            },
            json={
                "model": "gemma2:9b",
                "prompt": f"""
You are given an Italian phrase.
Determine its sentiment in form of emotion (joy, fear, apathetic, anger...).
Answer only with the emotion and nothing else.
This is the Italian phrase:
{result}""".strip(),
                "options": {
                    "temperature": 0.8
                },
                "stream": False,
            },
            timeout=30,
        )
        response.raise_for_status()
        return response.json()["response"]
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return "error"

In [ ]:
df["base_emotion"] = df["base"].apply(evaluate_prompt)

In [ ]:
df["joy_variant_emotion"] = df["joy"].apply(evaluate_prompt)